# 🍽️ Restaurant Finder — version Google Colab

Ce notebook recrée **exactement la même logique** que la CLI `restaurant-finder search` du projet (recherche OpenStreetMap + enrichissement Instagram + export CSV/Excel), mais **sans aucune installation locale** : tout s'exécute dans le navigateur (y compris sur mobile).

**Comment l'utiliser :**

1. Ouvre ce notebook sur [colab.research.google.com](https://colab.research.google.com) (depuis ton téléphone, `Fichier > Importer un notebook`, ou héberge-le sur GitHub/Drive pour l'ouvrir en un clic).
2. Menu `Exécution > Tout exécuter` (ou `Runtime > Run all`) — exécute toutes les cellules dans l'ordre, une seule fois par session.
3. Utilise ensuite la fonction `rechercher_restaurants("NomDeLaVille")` dans une nouvelle cellule, autant de fois que tu veux.

**Important :**

- Le runtime Colab est **éphémère** : si tu fermes l'onglet ou après un certain temps d'inactivité, tout est réinitialisé. Il faudra ré-exécuter toutes les cellules (`Tout exécuter`) à chaque nouvelle session.
- Pense à enregistrer une copie de ce notebook sur ton **Google Drive** (`Fichier > Enregistrer une copie dans Drive`) pour le retrouver facilement depuis ton téléphone pendant tes 2 semaines.
- Aucune clé API n'est nécessaire (OpenStreetMap, Nominatim, DuckDuckGo/`ddgs` sont gratuits et sans authentification).


In [ ]:
# @title Étape 1 — Installation des dépendances (identiques à pyproject.toml)
#
# On installe les mêmes contraintes de version que le fichier `pyproject.toml`
# du projet original. Certains paquets (typer, fastapi, uvicorn, beautifulsoup4)
# ne sont pas utilisés par ce notebook (ils servent à la CLI et au panel web,
# remplacés ici par la cellule Colab elle-même) mais on les installe quand même
# pour rester rigoureusement fidèle aux dépendances du projet.

%pip install -q \
    "typer>=0.12,<1.0" \
    "rich>=13.7,<14.0" \
    "requests>=2.31,<3.0" \
    "beautifulsoup4>=4.12,<5.0" \
    "rapidfuzz>=3.9,<4.0" \
    "pandas>=2.2,<3.0" \
    "openpyxl>=3.1,<4.0" \
    "pydantic>=2.7,<3.0" \
    "pydantic-settings>=2.3,<3.0" \
    "ddgs>=9.0,<10.0" \
    "fastapi>=0.110,<1.0" \
    "uvicorn>=0.29,<1.0"

print("✅ Dépendances installées.")


In [ ]:
# @title Étape 2 — Création de la structure du projet dans Colab
#
# On recrée ici l'arborescence exacte de `src/restaurant_finder/` du projet
# original, directement dans le système de fichiers éphémère de la VM Colab.
# Les cellules suivantes rempliront chaque fichier avec `%%writefile`.

from pathlib import Path

# Racine du projet recréé (persiste tant que le runtime Colab n'est pas réinitialisé).
PROJECT_ROOT = Path("/content/restaurant_finder")
SRC_ROOT = PROJECT_ROOT / "src" / "restaurant_finder"

PACKAGE_DIRS = [
    "",
    "domain",
    "geocoding",
    "sources",
    "filtering",
    "enrichment",
    "enrichment/search_providers",
    "export",
    "services",
    "cache",
    "http",
    "utils",
]
for relative_dir in PACKAGE_DIRS:
    (SRC_ROOT / relative_dir).mkdir(parents=True, exist_ok=True)

# Les `__init__.py` ne contiennent que de simples ré-exports (aucune logique
# métier) : on les génère ici en une seule fois pour ne pas multiplier les
# cellules `%%writefile` inutilement. Les modules avec de la vraie logique
# sont, eux, écrits dans leur propre cellule ci-dessous.
INIT_FILES = {
    "__init__.py": '''"""Restaurant Finder.

Outil de recherche automatisée de restaurants à partir d'OpenStreetMap,
avec enrichissement du profil Instagram officiel et export CSV/Excel.
"""

__version__ = "0.1.0"
''',
    "domain/__init__.py": '''from restaurant_finder.domain.models import BoundingBox, Restaurant

__all__ = ["Restaurant", "BoundingBox"]
''',
    "geocoding/__init__.py": '''from restaurant_finder.geocoding.nominatim_client import NominatimGeocoder

__all__ = ["NominatimGeocoder"]
''',
    "sources/__init__.py": '''from restaurant_finder.sources.base import RestaurantSource
from restaurant_finder.sources.overpass_source import OverpassRestaurantSource

__all__ = ["RestaurantSource", "OverpassRestaurantSource"]
''',
    "filtering/__init__.py": '''from restaurant_finder.filtering.chain_filter import (
    DEFAULT_CHAIN_PATTERNS,
    ChainRestaurantFilter,
)

__all__ = ["ChainRestaurantFilter", "DEFAULT_CHAIN_PATTERNS"]
''',
    "enrichment/__init__.py": '''from restaurant_finder.enrichment.instagram_finder import InstagramFinder

__all__ = ["InstagramFinder"]
''',
    # Note : contrairement au projet original, on n'expose pas ici
    # `DuckDuckGoSearchProvider` (scraping HTML), non utilisé par ce notebook
    # qui s'appuie uniquement sur `ddgs` (plus fiable, voir bootstrap.py).
    "enrichment/search_providers/__init__.py": '''from restaurant_finder.enrichment.search_providers.base import SearchProvider, SearchResult
from restaurant_finder.enrichment.search_providers.ddgs_provider import DdgsSearchProvider

__all__ = ["SearchProvider", "SearchResult", "DdgsSearchProvider"]
''',
    "export/__init__.py": '''from restaurant_finder.export.base import Exporter
from restaurant_finder.export.csv_exporter import CsvExporter
from restaurant_finder.export.excel_exporter import ExcelExporter

EXPORTERS: dict[str, Exporter] = {
    "csv": CsvExporter(),
    "xlsx": ExcelExporter(),
}

__all__ = ["Exporter", "CsvExporter", "ExcelExporter", "EXPORTERS"]
''',
    "services/__init__.py": '''from restaurant_finder.services.restaurant_finder_service import RestaurantFinderService

__all__ = ["RestaurantFinderService"]
''',
    "cache/__init__.py": '''from restaurant_finder.cache.file_cache import FileCache

__all__ = ["FileCache"]
''',
    "http/__init__.py": '''from restaurant_finder.http.client import build_http_session

__all__ = ["build_http_session"]
''',
    "utils/__init__.py": "",
}

for relative_path, content in INIT_FILES.items():
    (SRC_ROOT / relative_path).write_text(content, encoding="utf-8")

print(f"✅ Arborescence créée sous {SRC_ROOT}")


## Étape 3 — Recréation des modules du projet

Chaque cellule ci-dessous utilise la magie `%%writefile` pour écrire **le contenu exact** d'un fichier du projet original (`src/restaurant_finder/...`) dans l'environnement Colab. Exécute-les toutes, dans l'ordre.


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/exceptions.py
"""Exceptions métier de Restaurant Finder.

Centraliser les exceptions permet à la CLI (ou toute future couche de
présentation) de gérer les erreurs de façon uniforme, sans dépendre des
détails d'implémentation de chaque couche technique.
"""

from __future__ import annotations


class RestaurantFinderError(Exception):
    """Exception racine de l'application."""


class GeocodingError(RestaurantFinderError):
    """Levée quand une ville ne peut pas être géolocalisée."""


class LocationParsingError(RestaurantFinderError):
    """Levée quand un texte (coordonnées / lien Google Maps) est illisible."""


class RestaurantSourceError(RestaurantFinderError):
    """Levée quand une source de données de restaurants échoue."""


class SearchProviderError(RestaurantFinderError):
    """Levée quand un fournisseur de recherche web échoue."""


class ExportError(RestaurantFinderError):
    """Levée quand l'export d'un fichier (CSV, Excel...) échoue."""


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/config.py
"""Configuration centralisée de l'application.

Toutes les valeurs sont surchargeables via des variables d'environnement
préfixées par `RF_` (ou un fichier `.env` à la racine du projet), ce qui
évite de disséminer des constantes magiques dans le code métier.
"""

from __future__ import annotations

from pathlib import Path

from pydantic_settings import BaseSettings, SettingsConfigDict

#: Catégories OSM (tag `amenity`) supportées par défaut et leur libellé FR.
DEFAULT_CATEGORY_LABELS: dict[str, str] = {
    "restaurant": "Restaurant",
    "cafe": "Café",
    "fast_food": "Fast-food",
    "bar": "Bar",
    "pub": "Pub",
    "biergarten": "Biergarten",
}


class Settings(BaseSettings):
    """Paramètres de configuration, chargés depuis l'environnement / `.env`."""

    model_config = SettingsConfigDict(
        env_prefix="RF_",
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )

    # --- Réseau ---
    user_agent: str = "RestaurantFinder/0.1 (+https://github.com/example/restaurant-finder)"
    request_timeout_seconds: int = 30
    http_max_retries: int = 3
    http_backoff_factor: float = 1.0

    # --- Geocoding (Nominatim) ---
    nominatim_base_url: str = "https://nominatim.openstreetmap.org"
    nominatim_rate_limit_seconds: float = 1.0

    # --- Données restaurants (Overpass) ---
    # Les instances publiques sont souvent saturées (406 / timeouts). On essaie
    # plusieurs miroirs ; un miroir qui répond 200 avec une base de données
    # invalide / vide est écarté (ex. certains miroirs « fantômes »).
    # Les retries HTTP sur Overpass sont désactivés : un échec = miroir suivant.
    overpass_base_url: str = "https://overpass-api.de/api/interpreter"
    overpass_fallback_urls: tuple[str, ...] = (
        "https://z.overpass-api.de/api/interpreter",
        "https://lz4.overpass-api.de/api/interpreter",
    )
    overpass_timeout_seconds: int = 60
    overpass_connect_timeout_seconds: float = 10.0
    overpass_busy_retry_seconds: float = 8.0
    # Pause entre deux requêtes Overpass successives (recherche multi-points).
    overpass_rate_limit_seconds: float = 3.0
    # Par défaut : restaurants & cafés (les indépendants / bistrots ciblés).
    # Les bars / pubs / fast-food restent disponibles via --category.
    default_categories: tuple[str, ...] = ("restaurant", "cafe")
    # Rayon par défaut (mètres) autour de chaque point "pingué" (--near).
    default_search_radius_meters: float = 800.0

    # --- Enrichissement Instagram ---
    # workers=1 : les backends de recherche publics supportent mal le parallèle.
    instagram_search_enabled: bool = True
    instagram_search_max_workers: int = 1
    instagram_search_delay_seconds: float = 1.0
    instagram_match_threshold: int = 55  # score rapidfuzz (0-100)
    instagram_max_search_results: int = 10
    # Ne garder que les comptes avec strictement moins de N followers.
    instagram_max_followers: int = 1000
    instagram_filter_by_followers: bool = True
    # Si le nombre de followers est illisible (blocage IG), exclure le compte.
    instagram_exclude_unknown_followers: bool = True
    instagram_followers_delay_seconds: float = 1.5

    # --- Cache ---
    cache_enabled: bool = True
    cache_dir: Path = Path(".cache/restaurant_finder")
    cache_ttl_seconds: int = 7 * 24 * 60 * 60  # 7 jours

    # --- Export ---
    default_output_dir: Path = Path("output")


def get_settings() -> Settings:
    """Point d'accès unique à la configuration (facilite les tests via monkeypatch)."""

    return Settings()


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/domain/models.py
"""Modèles de domaine.

Ces modèles sont le cœur de l'application : ils ne dépendent d'aucune
librairie de scraping, d'export ou de CLI. Toutes les autres couches
convergent vers (ou partent de) `Restaurant`.
"""

from __future__ import annotations

from pydantic import BaseModel, Field


class BoundingBox(BaseModel):
    """Zone géographique rectangulaire (utilisée pour interroger Overpass)."""

    south: float
    north: float
    west: float
    east: float


class Restaurant(BaseModel):
    """Représente un restaurant (ou café, bar, fast-food...) identifié."""

    osm_id: str = Field(description="Identifiant unique OpenStreetMap, ex: 'node/12345'.")
    name: str
    category: str = Field(description="Libellé de catégorie en français, ex: 'Restaurant'.")
    address: str = ""
    city: str = ""
    latitude: float | None = None
    longitude: float | None = None
    brand: str | None = Field(
        default=None,
        description="Enseigne OSM (`brand`) si renseignée — utile pour détecter les chaînes.",
    )
    operator: str | None = Field(
        default=None,
        description="Opérateur OSM (`operator`) si renseigné.",
    )
    instagram_url: str | None = None
    instagram_followers: int | None = Field(
        default=None,
        description="Nombre de followers Instagram, si disponible.",
    )

    @property
    def instagram_handle(self) -> str | None:
        """Retourne uniquement le nom du compte Instagram (sans URL)."""

        # Import local pour éviter une dépendance circulaire domain ↔ enrichment.
        from restaurant_finder.enrichment.instagram_normalize import extract_handle

        return extract_handle(self.instagram_url)

    def to_export_row(self) -> dict[str, str]:
        """Convertit le restaurant en ligne prête pour l'export (CSV/Excel).

        Les noms de colonnes correspondent exactement au format de sortie
        attendu : Nom, Instagram, Adresse, Ville, Catégorie.
        La colonne Instagram contient le handle (ex: bistrot_le_cerey), pas l'URL.
        """

        return {
            "Nom": self.name,
            "Instagram": self.instagram_handle or "",
            "Adresse": self.address,
            "Ville": self.city,
            "Catégorie": self.category,
        }


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/utils/text.py
"""Fonctions utilitaires de normalisation de texte.

Utilisées à la fois pour la construction d'adresses et pour le matching
flou entre le nom d'un restaurant et les résultats de recherche web.
"""

from __future__ import annotations

import re
import unicodedata


def strip_accents(text: str) -> str:
    """Retire les accents d'une chaîne (é -> e, à -> a, ...)."""

    normalized = unicodedata.normalize("NFKD", text)
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_text(text: str) -> str:
    """Normalise un texte pour le comparer de façon robuste.

    - minuscule
    - sans accents
    - sans ponctuation
    - espaces multiples réduits à un seul
    """

    text = strip_accents(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def join_non_empty(parts: list[str | None], separator: str = ", ") -> str:
    """Joint les éléments non vides d'une liste avec un séparateur."""

    return separator.join(part.strip() for part in parts if part and part.strip())


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/utils/logging.py
"""Configuration du logging applicatif."""

from __future__ import annotations

import logging

from rich.logging import RichHandler


def setup_logging(verbose: bool = False) -> None:
    """Configure un logging homogène et lisible sur toute l'application.

    Args:
        verbose: si True, active le niveau DEBUG (sinon INFO).
    """

    level = logging.DEBUG if verbose else logging.INFO

    logging.basicConfig(
        level=level,
        format="%(message)s",
        datefmt="[%X]",
        handlers=[RichHandler(rich_tracebacks=True, show_path=verbose)],
        force=True,
    )

    # Bibliothèques tierces trop verbeuses : on les calme, sauf en mode verbose.
    if not verbose:
        for noisy_logger in ("urllib3", "requests", "ddgs", "httpx", "httpcore"):
            logging.getLogger(noisy_logger).setLevel(logging.WARNING)


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/cache/file_cache.py
"""Cache disque simple, à base de fichiers JSON avec expiration (TTL).

Objectif : éviter de re-solliciter Nominatim / Overpass / DuckDuckGo à
chaque exécution pour les mêmes requêtes, ce qui est à la fois plus
rapide et plus respectueux des services publics gratuits utilisés.

Volontairement minimaliste (pas de dépendance externe type `diskcache`)
car le cahier des charges privilégie une infrastructure légère.
"""

from __future__ import annotations

import hashlib
import json
import logging
import time
from pathlib import Path
from typing import Any

logger = logging.getLogger(__name__)


class FileCache:
    """Cache clé/valeur persistant sur disque, avec durée de vie (TTL)."""

    def __init__(self, cache_dir: Path, ttl_seconds: int, enabled: bool = True) -> None:
        self._cache_dir = cache_dir
        self._ttl_seconds = ttl_seconds
        self._enabled = enabled
        if self._enabled:
            self._cache_dir.mkdir(parents=True, exist_ok=True)

    def _path_for(self, key: str) -> Path:
        digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
        return self._cache_dir / f"{digest}.json"

    def get(self, key: str) -> Any | None:
        """Retourne la valeur en cache pour `key`, ou None si absente/expirée."""

        if not self._enabled:
            return None

        path = self._path_for(key)
        if not path.exists():
            return None

        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as exc:
            logger.debug("Cache illisible pour %s (%s), on l'ignore.", key, exc)
            return None

        if time.time() > payload.get("expires_at", 0):
            path.unlink(missing_ok=True)
            return None

        return payload.get("value")

    def set(self, key: str, value: Any) -> None:
        """Enregistre `value` en cache pour `key`, avec le TTL configuré."""

        if not self._enabled:
            return

        path = self._path_for(key)
        payload = {"expires_at": time.time() + self._ttl_seconds, "value": value}
        try:
            path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
        except OSError as exc:
            logger.debug("Impossible d'écrire le cache pour %s (%s).", key, exc)


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/http/client.py
"""Fabrique de session HTTP partagée, avec retries et User-Agent identifié.

Nominatim et Overpass exigent tous deux un User-Agent explicite (leurs
politiques d'usage interdisent le User-Agent par défaut des librairies
HTTP). Centraliser la création de la session garantit que toutes les
requêtes sortantes respectent cette règle.
"""

from __future__ import annotations

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from restaurant_finder.config import Settings


def build_http_session(settings: Settings, *, max_retries: int | None = None) -> requests.Session:
    """Construit une `requests.Session` configurée pour l'application.

    Args:
        settings: configuration applicative.
        max_retries: surcharge optionnelle du nombre de retries HTTP.
            Passer `0` pour Overpass (le basculement de miroir doit être immédiat).
    """

    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": settings.user_agent,
            "Accept": "application/json",
        }
    )

    retries = settings.http_max_retries if max_retries is None else max_retries
    retry_strategy = Retry(
        total=retries,
        backoff_factor=settings.http_backoff_factor,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET", "POST"),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/geocoding/geo_math.py
"""Calculs géométriques simples pour les recherches centrées sur un point.

Approximations volontairement légères (pas de dépendance type geopy) :
suffisantes pour construire une bounding box autour d'un point GPS et
pour filtrer des résultats par distance à vol d'oiseau.
"""

from __future__ import annotations

import math
from typing import NamedTuple

from restaurant_finder.domain.models import BoundingBox

#: Mètres par degré de latitude (quasi constant sur Terre).
_METERS_PER_DEGREE_LATITUDE = 111_320.0
_EARTH_RADIUS_METERS = 6_371_000.0


class PointQuery(NamedTuple):
    """Un lieu "pingué" (ex: sur Google Maps ou sur la carte du panel web).

    Chaque point porte son propre rayon, ce qui permet à l'utilisateur de
    cibler précisément une zone dense avec un petit rayon et une zone plus
    large avec un rayon plus grand, dans la même recherche.
    """

    latitude: float
    longitude: float
    radius_meters: float


def bbox_from_point(latitude: float, longitude: float, radius_meters: float) -> BoundingBox:
    """Construit une bounding box carrée englobant un cercle de rayon donné."""

    lat_delta = radius_meters / _METERS_PER_DEGREE_LATITUDE
    lon_scale = max(math.cos(math.radians(latitude)), 1e-6)
    lon_delta = radius_meters / (_METERS_PER_DEGREE_LATITUDE * lon_scale)

    return BoundingBox(
        south=latitude - lat_delta,
        north=latitude + lat_delta,
        west=longitude - lon_delta,
        east=longitude + lon_delta,
    )


def haversine_distance_meters(
    latitude1: float, longitude1: float, latitude2: float, longitude2: float
) -> float:
    """Distance à vol d'oiseau (en mètres) entre deux points GPS."""

    phi1, phi2 = math.radians(latitude1), math.radians(latitude2)
    delta_phi = math.radians(latitude2 - latitude1)
    delta_lambda = math.radians(longitude2 - longitude1)

    a = (
        math.sin(delta_phi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    )
    return 2 * _EARTH_RADIUS_METERS * math.asin(math.sqrt(a))


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/geocoding/location_parser.py
"""Interprétation d'une localisation fournie par l'utilisateur.

Objectif produit : permettre de « pinguer » un lieu sur Google Maps et de
coller directement ce que Google Maps propose de copier, à savoir :

- des coordonnées brutes, ex: "45.917707, 6.131942" (clic droit -> le
  premier élément du menu copie exactement ce format) ;
- une URL Google Maps contenant les coordonnées dans son chemin
  (`/@lat,lon,zoom`) ou ses paramètres (`?q=lat,lon`, `&ll=lat,lon`) ;
- un lien court (`https://maps.app.goo.gl/...`), résolu via une requête
  HTTP pour retrouver l'URL complète.
"""

from __future__ import annotations

import logging
import re

import requests

from restaurant_finder.config import Settings
from restaurant_finder.exceptions import LocationParsingError

logger = logging.getLogger(__name__)

_PLAIN_COORDINATES = re.compile(
    r"^\s*(-?\d{1,3}(?:\.\d+)?)\s*,\s*(-?\d{1,3}(?:\.\d+)?)\s*$"
)
_URL_AT_PATTERN = re.compile(r"@(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)")
_URL_QUERY_PATTERNS = (
    re.compile(r"[?&]q=(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)"),
    re.compile(r"[?&]query=(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)"),
    re.compile(r"[?&]ll=(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)"),
)
class LocationInputParser:
    """Convertit un texte (coordonnées ou lien Google Maps) en (latitude, longitude)."""

    def __init__(self, session: requests.Session, settings: Settings) -> None:
        self._session = session
        self._settings = settings

    def parse(self, raw_input: str) -> tuple[float, float]:
        """Retourne (latitude, longitude), ou lève `LocationParsingError`."""

        text = raw_input.strip()
        if not text:
            raise LocationParsingError("Valeur vide.")

        coordinates = self._try_plain_coordinates(text)
        if coordinates is not None:
            return coordinates

        coordinates = self._try_url_patterns(text)
        if coordinates is not None:
            return coordinates

        if self._looks_like_url(text):
            resolved_url = self._resolve_redirect(text)
            if resolved_url and resolved_url != text:
                coordinates = self._try_url_patterns(resolved_url)
                if coordinates is not None:
                    return coordinates

        raise LocationParsingError(
            f"Impossible d'interpréter {raw_input!r} comme des coordonnées GPS ou un lien "
            "Google Maps. Copie soit les coordonnées ('45.9177, 6.1319'), soit l'URL complète."
        )

    @staticmethod
    def _try_plain_coordinates(text: str) -> tuple[float, float] | None:
        match = _PLAIN_COORDINATES.match(text)
        if not match:
            return None
        return _to_coordinates(match.group(1), match.group(2))

    @staticmethod
    def _try_url_patterns(text: str) -> tuple[float, float] | None:
        at_match = _URL_AT_PATTERN.search(text)
        if at_match:
            return _to_coordinates(at_match.group(1), at_match.group(2))

        for pattern in _URL_QUERY_PATTERNS:
            match = pattern.search(text)
            if match:
                return _to_coordinates(match.group(1), match.group(2))

        return None

    @staticmethod
    def _looks_like_url(text: str) -> bool:
        return text.lower().startswith(("http://", "https://"))

    def _resolve_redirect(self, url: str) -> str | None:
        """Suit les redirections d'un lien court Google Maps pour révéler les coordonnées."""

        try:
            response = self._session.get(
                url,
                timeout=self._settings.request_timeout_seconds,
                allow_redirects=True,
            )
        except requests.RequestException as exc:
            logger.debug("Résolution du lien %r échouée : %s", url, exc)
            return None

        return response.url


def _to_coordinates(raw_latitude: str, raw_longitude: str) -> tuple[float, float]:
    latitude, longitude = float(raw_latitude), float(raw_longitude)
    if not (-90 <= latitude <= 90) or not (-180 <= longitude <= 180):
        raise LocationParsingError(
            f"Coordonnées hors limites : ({latitude}, {longitude})."
        )
    return latitude, longitude


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/geocoding/nominatim_client.py
"""Client de géocodage basé sur Nominatim (OpenStreetMap).

Rôle unique : transformer un nom de ville en zone géographique (bbox)
exploitable par la source Overpass. Isolé dans sa propre classe pour
pouvoir, demain, être remplacé par un autre fournisseur de géocodage
sans impacter le reste de l'application.
"""

from __future__ import annotations

import logging
import time

import requests

from restaurant_finder.cache import FileCache
from restaurant_finder.config import Settings
from restaurant_finder.domain.models import BoundingBox
from restaurant_finder.exceptions import GeocodingError

logger = logging.getLogger(__name__)


class NominatimGeocoder:
    """Géocode un nom de ville en `BoundingBox` via l'API Nominatim."""

    def __init__(
        self,
        session: requests.Session,
        settings: Settings,
        cache: FileCache | None = None,
    ) -> None:
        self._session = session
        self._settings = settings
        self._cache = cache
        self._last_request_time: float = 0.0

    def _respect_rate_limit(self) -> None:
        """Nominatim impose un maximum de 1 requête/seconde."""

        elapsed = time.monotonic() - self._last_request_time
        wait_time = self._settings.nominatim_rate_limit_seconds - elapsed
        if wait_time > 0:
            time.sleep(wait_time)

    def geocode_city(self, city: str) -> BoundingBox:
        """Retourne la zone géographique correspondant à `city`.

        Raises:
            GeocodingError: si la ville est introuvable ou en cas d'erreur réseau.
        """

        cache_key = f"geocode:{city.strip().lower()}"
        if self._cache is not None:
            cached = self._cache.get(cache_key)
            if cached is not None:
                logger.debug("Bounding box pour %r trouvée en cache.", city)
                return BoundingBox(**cached)

        self._respect_rate_limit()
        self._last_request_time = time.monotonic()

        try:
            response = self._session.get(
                f"{self._settings.nominatim_base_url}/search",
                params={"city": city, "format": "jsonv2", "limit": "1"},
                timeout=self._settings.request_timeout_seconds,
            )
            response.raise_for_status()
            results = response.json()
        except (requests.RequestException, ValueError) as exc:
            raise GeocodingError(f"Échec du géocodage de la ville '{city}': {exc}") from exc

        if not results:
            raise GeocodingError(f"Ville introuvable : '{city}'.")

        raw_bbox = results[0].get("boundingbox")
        if not raw_bbox or len(raw_bbox) != 4:
            raise GeocodingError(f"Réponse Nominatim invalide pour '{city}'.")

        south, north, west, east = (float(value) for value in raw_bbox)
        bbox = BoundingBox(south=south, north=north, west=west, east=east)

        if self._cache is not None:
            self._cache.set(cache_key, bbox.model_dump())

        return bbox

    def reverse_geocode_city(self, latitude: float, longitude: float) -> str | None:
        """Retourne un nom de localité (ville/village) proche du point donné.

        Utilisé pour étiqueter la colonne "Ville" des recherches par point GPS
        (`--near`), où il n'y a pas de nom de ville fourni explicitement.
        Best-effort : retourne None si Nominatim échoue ou ne trouve rien.
        """

        cache_key = f"reverse:{round(latitude, 5)}:{round(longitude, 5)}"
        if self._cache is not None:
            cached = self._cache.get(cache_key)
            if cached is not None:
                return cached or None

        self._respect_rate_limit()
        self._last_request_time = time.monotonic()

        try:
            response = self._session.get(
                f"{self._settings.nominatim_base_url}/reverse",
                params={
                    "lat": f"{latitude:.6f}",
                    "lon": f"{longitude:.6f}",
                    "format": "jsonv2",
                    "zoom": "14",
                },
                timeout=self._settings.request_timeout_seconds,
            )
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            logger.debug("Reverse géocodage échoué pour (%s, %s) : %s", latitude, longitude, exc)
            if self._cache is not None:
                self._cache.set(cache_key, "")
            return None

        address = payload.get("address", {})
        label = (
            address.get("city")
            or address.get("town")
            or address.get("village")
            or address.get("municipality")
            or address.get("suburb")
        )

        if self._cache is not None:
            self._cache.set(cache_key, label or "")

        return label


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/sources/base.py
"""Interface abstraite d'une source de données de restaurants.

Toute nouvelle source (Google Places, Yelp, import CSV manuel...) doit
implémenter ce contrat pour être utilisable par `RestaurantFinderService`
sans aucune modification du reste du code (principe ouvert/fermé).
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from collections.abc import Sequence

from restaurant_finder.domain.models import Restaurant
from restaurant_finder.geocoding.geo_math import PointQuery


class RestaurantSource(ABC):
    """Contrat pour toute source capable de lister des restaurants."""

    @abstractmethod
    def find_restaurants(self, city: str, categories: Sequence[str]) -> list[Restaurant]:
        """Retourne les restaurants de `city` correspondant à `categories`.

        Args:
            city: nom de la ville à rechercher.
            categories: liste de catégories OSM (ex: "restaurant", "cafe").
        """
        raise NotImplementedError

    @abstractmethod
    def find_restaurants_near_points(
        self,
        points: Sequence[PointQuery],
        categories: Sequence[str],
    ) -> list[Restaurant]:
        """Retourne les restaurants situés dans le rayon de chaque point.

        Args:
            points: lieux "pingués" (latitude, longitude, rayon en mètres).
                Plusieurs points étendent la zone de recherche à plusieurs
                endroits distincts, chacun avec son propre rayon.
            categories: liste de catégories OSM (ex: "restaurant", "cafe").
        """
        raise NotImplementedError


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/sources/overpass_source.py
"""Source de restaurants basée sur l'API Overpass (données OpenStreetMap).

Deux modes de recherche :
1. Par ville : nom de ville -> bounding box (Nominatim) -> requête Overpass.
2. Par point GPS ("pin" Google Maps) : point + rayon -> bounding box ->
   requête Overpass -> filtrage précis par distance à vol d'oiseau.
   Plusieurs points peuvent être combinés pour étendre la recherche.
"""

from __future__ import annotations

import logging
import re
import time
from collections.abc import Sequence

import requests

from restaurant_finder.config import DEFAULT_CATEGORY_LABELS, Settings
from restaurant_finder.domain.models import BoundingBox, Restaurant
from restaurant_finder.enrichment.instagram_normalize import to_profile_url
from restaurant_finder.exceptions import RestaurantSourceError
from restaurant_finder.geocoding.geo_math import (
    PointQuery,
    bbox_from_point,
    haversine_distance_meters,
)
from restaurant_finder.geocoding.nominatim_client import NominatimGeocoder
from restaurant_finder.sources.base import RestaurantSource
from restaurant_finder.utils.text import join_non_empty

logger = logging.getLogger(__name__)

_OVERPASS_QUERY_TEMPLATE = """
[out:json][timeout:{timeout}];
(
  node["amenity"~"^({categories})$"]({south},{west},{north},{east});
  way["amenity"~"^({categories})$"]({south},{west},{north},{east});
  relation["amenity"~"^({categories})$"]({south},{west},{north},{east});
);
out center tags;
"""

#: Un timestamp OSM valide ressemble à "2026-07-29T22:17:47Z".
_VALID_OSM_TIMESTAMP = re.compile(r"^\d{4}-\d{2}-\d{2}T")


class OverpassRestaurantSource(RestaurantSource):
    """Récupère les restaurants d'une zone via Overpass (OpenStreetMap)."""

    def __init__(
        self,
        session: requests.Session,
        settings: Settings,
        geocoder: NominatimGeocoder,
    ) -> None:
        self._session = session
        self._settings = settings
        self._geocoder = geocoder

    def find_restaurants(self, city: str, categories: Sequence[str]) -> list[Restaurant]:
        bbox = self._geocoder.geocode_city(city)
        restaurants = self._search_bbox(bbox, categories, fallback_city=city)
        logger.info("%d établissement(s) trouvé(s) à %s.", len(restaurants), city)
        return restaurants

    def find_restaurants_near_points(
        self,
        points: Sequence[PointQuery],
        categories: Sequence[str],
    ) -> list[Restaurant]:
        restaurants: list[Restaurant] = []
        seen_osm_ids: set[str] = set()

        for index, point in enumerate(points, start=1):
            bbox = bbox_from_point(point.latitude, point.longitude, point.radius_meters)
            label = (
                self._geocoder.reverse_geocode_city(point.latitude, point.longitude)
                or f"Lieu {index}"
            )

            found_here = 0
            for restaurant in self._search_bbox(bbox, categories, fallback_city=label):
                if restaurant.osm_id in seen_osm_ids:
                    continue

                if restaurant.latitude is not None and restaurant.longitude is not None:
                    distance = haversine_distance_meters(
                        point.latitude, point.longitude, restaurant.latitude, restaurant.longitude
                    )
                    if distance > point.radius_meters:
                        continue

                seen_osm_ids.add(restaurant.osm_id)
                restaurants.append(restaurant)
                found_here += 1

            logger.info(
                "%d établissement(s) trouvé(s) autour de (%.5f, %.5f) [%s, rayon %.0fm].",
                found_here,
                point.latitude,
                point.longitude,
                label,
                point.radius_meters,
            )

            if index < len(points):
                time.sleep(self._settings.overpass_rate_limit_seconds)

        return restaurants

    def _search_bbox(
        self, bbox: BoundingBox, categories: Sequence[str], fallback_city: str
    ) -> list[Restaurant]:
        elements = self._query_overpass(bbox, categories)

        restaurants: list[Restaurant] = []
        skipped = 0
        for element in elements:
            restaurant = self._parse_element(element, fallback_city=fallback_city)
            if restaurant is None:
                skipped += 1
                continue
            restaurants.append(restaurant)

        if skipped:
            logger.debug("%d éléments OSM ignorés (pas de nom exploitable).", skipped)

        return restaurants

    def _query_overpass(self, bbox: BoundingBox, categories: Sequence[str]) -> list[dict]:
        categories_pattern = "|".join(categories)
        query = _OVERPASS_QUERY_TEMPLATE.format(
            timeout=self._settings.overpass_timeout_seconds,
            categories=categories_pattern,
            south=bbox.south,
            west=bbox.west,
            north=bbox.north,
            east=bbox.east,
        )

        endpoints = (self._settings.overpass_base_url, *self._settings.overpass_fallback_urls)
        timeout = (
            self._settings.overpass_connect_timeout_seconds,
            self._settings.overpass_timeout_seconds,
        )
        last_error: Exception | None = None

        for endpoint in endpoints:
            try:
                elements = self._query_endpoint(endpoint, query, timeout)
            except (requests.RequestException, ValueError, RestaurantSourceError) as exc:
                logger.warning("Miroir Overpass indisponible (%s) : %s", endpoint, exc)
                last_error = exc
                continue

            return elements

        raise RestaurantSourceError(
            "Échec de la requête Overpass sur tous les miroirs disponibles. "
            "Les serveurs OpenStreetMap publics sont probablement saturés : "
            f"réessaie dans 1–2 minutes. Détail : {last_error}"
        ) from last_error

    def _query_endpoint(
        self,
        endpoint: str,
        query: str,
        timeout: tuple[float, float],
    ) -> list[dict]:
        """Interroge un miroir ; en cas de saturation (406/429), réessaie une fois."""

        attempts = 2
        for attempt in range(1, attempts + 1):
            logger.info(
                "Interrogation Overpass via %s (essai %d/%d)...",
                endpoint,
                attempt,
                attempts,
            )
            response = self._session.post(endpoint, data={"data": query}, timeout=timeout)

            if response.status_code in {406, 429, 504} and attempt < attempts:
                wait = self._settings.overpass_busy_retry_seconds
                logger.warning(
                    "Miroir %s saturé (HTTP %d). Nouvelle tentative dans %.0fs...",
                    endpoint,
                    response.status_code,
                    wait,
                )
                time.sleep(wait)
                continue

            response.raise_for_status()
            payload = response.json()
            self._ensure_payload_is_usable(endpoint, payload)
            return payload.get("elements", [])

        raise RestaurantSourceError(f"Miroir Overpass saturé : {endpoint}")

    @staticmethod
    def _ensure_payload_is_usable(endpoint: str, payload: dict) -> None:
        """Écarte les miroirs qui répondent 200 avec une base OSM invalide."""

        timestamp = str((payload.get("osm3s") or {}).get("timestamp_osm_base") or "")
        if not _VALID_OSM_TIMESTAMP.match(timestamp):
            raise RestaurantSourceError(
                f"Réponse Overpass invalide sur {endpoint} "
                f"(timestamp_osm_base={timestamp!r}). Miroir probablement hors service."
            )

    @staticmethod
    def _parse_element(element: dict, fallback_city: str) -> Restaurant | None:
        tags: dict = element.get("tags", {})
        name = tags.get("name")
        if not name:
            return None

        latitude = element.get("lat")
        longitude = element.get("lon")
        if latitude is None or longitude is None:
            center = element.get("center") or {}
            latitude = center.get("lat")
            longitude = center.get("lon")

        amenity = tags.get("amenity", "")
        category = DEFAULT_CATEGORY_LABELS.get(amenity, amenity.replace("_", " ").capitalize())

        street_line = join_non_empty([tags.get("addr:housenumber"), tags.get("addr:street")], " ")
        address = join_non_empty([street_line, tags.get("addr:postcode")])

        city = tags.get("addr:city") or fallback_city

        osm_id = f"{element.get('type', 'node')}/{element.get('id')}"
        instagram_url = to_profile_url(
            tags.get("contact:instagram") or tags.get("instagram")
        )

        return Restaurant(
            osm_id=osm_id,
            name=name,
            category=category,
            address=address,
            city=city,
            latitude=latitude,
            longitude=longitude,
            brand=tags.get("brand"),
            operator=tags.get("operator"),
            instagram_url=instagram_url,
        )


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/filtering/chain_filter.py
"""Liste et détection des chaînes / franchises à exclure.

Objectif produit : ne garder que les restaurants et bistrots indépendants,
pas les grandes enseignes (McDo, Subway, Burger King, etc.).

Le filtre s'appuie sur :
1. le nom de l'établissement
2. les tags OSM `brand` / `operator` quand ils sont disponibles
"""

from __future__ import annotations

import re

from restaurant_finder.domain.models import Restaurant
from restaurant_finder.utils.text import normalize_text

#: Motifs normalisés (sans accents, minuscules) représentant des enseignes.
#: Suffisamment spécifiques pour limiter les faux positifs.
DEFAULT_CHAIN_PATTERNS: tuple[str, ...] = (
    # Fast-food internationaux
    r"\bmcdonald",
    r"\bmc ?do\b",
    r"\bburger ?king\b",
    r"\bkfc\b",
    r"\bsubway\b",
    r"\bquick\b",
    r"\bfive ?guys\b",
    r"\bwendy'?s\b",
    r"\btaco ?bell\b",
    r"\bnando'?s\b",
    r"\bchipotle\b",
    r"\bpopeyes\b",
    r"\bdunkin\b",
    r"\btim ?hortons\b",
    r"\bkrispy ?kreme\b",
    r"\bsteak ?n ?shake\b",
    # Pizza / burgers / tacos FR & international
    r"\bdomino'?s\b",
    r"\bpizza ?hut\b",
    r"\bpapa ?john",
    r"\bpizza ?pai\b",
    r"\bdel ?arte\b",
    r"\bspeed ?rabbit\b",
    r"\bla boite a pizza\b",
    r"\bo'?tacos\b",
    r"\bbig ?fernand\b",
    r"\bking ?marcel\b",
    r"\bbuffalo ?grill\b",
    r"\bhippopotamus\b",
    r"\bcourtepaille\b",
    r"\bflunch\b",
    r"\bindiana ?cafe\b",
    r"\bleon de bruxelles\b",
    r"\bhard ?rock\b",
    r"\bvapiano\b",
    r"\btgi ?friday",
    r"\bchicken ?street\b",
    r"\bfresh burritos\b",
    r"\bbearburger\b",
    # Café / boulangerie chaînes
    r"\bstarbucks\b",
    r"\bcosta ?coffee\b",
    r"\bcolumbus ?cafe\b",
    r"\bbagelstein\b",
    r"\bbrioche doree\b",
    r"\bla mie caline\b",
    r"\bboulangerie paul\b",
    r"\bpaul bakery\b",
    r"\bpomme de pain\b",
    r"\beric kayser\b",
    r"\bexki\b",
    r"\bcojean\b",
    r"\bpret a manger\b",
    r"\bclass'?croute\b",
    # Asiatique / sushis chaînes
    r"\bsushi ?shop\b",
    r"\bplanet ?sushi\b",
    r"\bsushi ?daily\b",
    r"\bpitaya\b",
    r"\bpokawa\b",
    r"\bmezzo di pasta\b",
    r"\bwok to walk\b",
    r"\byum yum\b",
    # Autres enseignes courantes
    r"\bbert'?s\b",
    r"\bautogrill\b",
    r"\brelay\b",
    r"\bcasino cafeteria\b",
)


class ChainRestaurantFilter:
    """Filtre les établissements appartenant à une chaîne / franchise connue."""

    def __init__(self, patterns: tuple[str, ...] | None = None) -> None:
        raw_patterns = patterns if patterns is not None else DEFAULT_CHAIN_PATTERNS
        self._compiled = [re.compile(pattern, re.IGNORECASE) for pattern in raw_patterns]

    def is_chain(self, restaurant: Restaurant) -> bool:
        """Retourne True si l'établissement ressemble à une enseigne connue."""

        haystacks = [
            normalize_text(restaurant.name),
            normalize_text(restaurant.brand or ""),
            normalize_text(restaurant.operator or ""),
        ]
        for haystack in haystacks:
            if not haystack:
                continue
            for pattern in self._compiled:
                if pattern.search(haystack):
                    return True
        return False

    def exclude_chains(self, restaurants: list[Restaurant]) -> tuple[list[Restaurant], int]:
        """Retourne (indépendants, nombre_exclus)."""

        independents: list[Restaurant] = []
        excluded = 0
        for restaurant in restaurants:
            if self.is_chain(restaurant):
                excluded += 1
            else:
                independents.append(restaurant)
        return independents, excluded


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/enrichment/matching.py
"""Logique de matching flou entre un nom de restaurant et un résultat web.

Isolée dans son propre module car c'est la partie la plus "heuristique"
du projet : elle est testée unitairement en isolation, indépendamment
du réseau.
"""

from __future__ import annotations

from rapidfuzz import fuzz

from restaurant_finder.utils.text import normalize_text


def score_candidate(restaurant_name: str, candidate_text: str) -> int:
    """Retourne un score de similarité (0-100) entre un restaurant et un texte.

    Combine deux métriques rapidfuzz pour être robuste à la fois aux
    réordonnancements de mots et aux textes partiels (ex: "Le Petit Café"
    vs "lepetitcafe_officiel · Instagram").
    """

    normalized_name = normalize_text(restaurant_name)
    normalized_candidate = normalize_text(candidate_text)

    if not normalized_name or not normalized_candidate:
        return 0

    token_score = fuzz.token_sort_ratio(normalized_name, normalized_candidate)
    partial_score = fuzz.partial_ratio(normalized_name, normalized_candidate)

    return round(max(token_score, partial_score))


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/enrichment/instagram_normalize.py
"""Normalisation des identifiants / URLs Instagram.

Les tags OSM et les résultats de recherche renvoient des formats
hétérogènes (@handle, URL complète, chemin /popular/, etc.).
Ce module centralise la conversion vers une URL de profil canonique.
"""

from __future__ import annotations

import re
from urllib.parse import urlparse

_INSTAGRAM_HOST_PATTERN = re.compile(r"(^|\.)instagram\.com$", re.IGNORECASE)
_RESERVED_PATH_SEGMENTS = {
    "p",
    "reel",
    "reels",
    "explore",
    "accounts",
    "stories",
    "directory",
    "developer",
    "about",
    "legal",
    "tv",
    "popular",
    "tags",
    "locations",
}
_HANDLE_PATTERN = re.compile(r"^[A-Za-z0-9._]{1,30}$")
_HANDLE_FROM_PATH = re.compile(r"^/([A-Za-z0-9._]+)/?")
_HANDLE_FROM_TEXT = re.compile(
    r"(?:https?://)?(?:www\.)?instagram\.com/([A-Za-z0-9._]+)/?",
    re.IGNORECASE,
)


def extract_handle(value: str | None) -> str | None:
    """Extrait un handle Instagram depuis une URL, un @handle ou un texte libre."""

    if not value:
        return None

    text = value.strip()
    if not text:
        return None

    if text.startswith("@"):
        text = text[1:].strip()

    match = _HANDLE_FROM_TEXT.search(text)
    if match:
        handle = match.group(1)
        if handle.lower() not in _RESERVED_PATH_SEGMENTS and _HANDLE_PATTERN.match(handle):
            return handle

    parsed = urlparse(text if "://" in text else f"https://{text}")
    if _INSTAGRAM_HOST_PATTERN.search(parsed.netloc):
        path_match = _HANDLE_FROM_PATH.match(parsed.path)
        if path_match:
            handle = path_match.group(1)
            if handle.lower() not in _RESERVED_PATH_SEGMENTS and _HANDLE_PATTERN.match(handle):
                return handle
        return None

    # Valeur OSM du type "mon_resto_nice" sans URL.
    candidate = text.strip("/")
    if _HANDLE_PATTERN.match(candidate) and candidate.lower() not in _RESERVED_PATH_SEGMENTS:
        return candidate

    return None


def to_profile_url(value: str | None) -> str | None:
    """Convertit une valeur Instagram quelconque en URL de profil canonique."""

    handle = extract_handle(value)
    if handle is None:
        return None
    return f"https://www.instagram.com/{handle}/"


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/enrichment/search_providers/base.py
"""Interface abstraite d'un fournisseur de recherche web.

Il n'existe pas d'API gratuite officielle de recherche web permettant de
retrouver un profil Instagram. Cette interface isole ce point de risque :
`DdgsSearchProvider` (via la librairie `ddgs`) peut être remplacé par un
fournisseur payant plus fiable (SerpApi, Google Custom Search...) en
implémentant simplement ce même contrat.
"""

from __future__ import annotations

from abc import ABC, abstractmethod

from pydantic import BaseModel


class SearchResult(BaseModel):
    """Un résultat de recherche web générique."""

    title: str
    url: str
    snippet: str = ""


class SearchProvider(ABC):
    """Contrat pour tout fournisseur de recherche web."""

    @abstractmethod
    def search(self, query: str, max_results: int) -> list[SearchResult]:
        """Exécute une recherche web et retourne jusqu'à `max_results` résultats."""
        raise NotImplementedError


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/enrichment/search_providers/ddgs_provider.py
"""Fournisseur de recherche web basé sur la librairie `ddgs` (ex duckduckgo-search).

Contrairement au scraping HTML direct de DuckDuckGo (souvent bloqué par
anti-bot HTTP 202), `ddgs` agrège plusieurs backends de recherche et
reste utilisable sans clé API — adapté à la découverte de profils
Instagram.
"""

from __future__ import annotations

import logging
import threading

from ddgs import DDGS

from restaurant_finder.config import Settings
from restaurant_finder.enrichment.search_providers.base import SearchProvider, SearchResult
from restaurant_finder.exceptions import SearchProviderError

logger = logging.getLogger(__name__)


class DdgsSearchProvider(SearchProvider):
    """Recherche web via la librairie `ddgs` (sans clé API)."""

    def __init__(self, settings: Settings) -> None:
        self._settings = settings
        self._lock = threading.Lock()

    def search(self, query: str, max_results: int) -> list[SearchResult]:
        try:
            # Un seul appel réseau à la fois : les backends publics sont sensibles
            # au parallélisme agressif et peuvent renvoyer des pages vides.
            with self._lock, DDGS() as client:
                raw_results = list(client.text(query, max_results=max_results))
        except Exception as exc:  # noqa: BLE001 - API tierce hétérogène
            raise SearchProviderError(
                f"Échec de la recherche ddgs pour {query!r}: {exc}"
            ) from exc

        results: list[SearchResult] = []
        for item in raw_results:
            url = str(item.get("href") or item.get("link") or "").strip()
            title = str(item.get("title") or "").strip()
            snippet = str(item.get("body") or item.get("description") or "").strip()
            if not url:
                continue
            results.append(SearchResult(title=title, url=url, snippet=snippet))

        if not results:
            logger.debug("ddgs : aucun résultat pour %r.", query)

        return results


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/enrichment/instagram_finder.py
"""Recherche du profil Instagram officiel d'un restaurant.

Priorité :
1. Si OpenStreetMap fournit déjà un tag Instagram → on l'utilise tel quel.
2. Sinon, recherche web (DuckDuckGo) avec plusieurs requêtes successives.
3. Matching flou (rapidfuzz) pour retenir le profil le plus crédible.
"""

from __future__ import annotations

import logging
import threading
import time

from restaurant_finder.cache import FileCache
from restaurant_finder.config import Settings
from restaurant_finder.domain.models import Restaurant
from restaurant_finder.enrichment.instagram_normalize import extract_handle, to_profile_url
from restaurant_finder.enrichment.matching import score_candidate
from restaurant_finder.enrichment.search_providers.base import SearchProvider, SearchResult
from restaurant_finder.exceptions import SearchProviderError

logger = logging.getLogger(__name__)


class InstagramFinder:
    """Tente de retrouver le profil Instagram officiel d'un restaurant.

    Thread-safe : plusieurs threads peuvent appeler `find()` en parallèle
    (voir `RestaurantFinderService`), le rate limiter interne garantit un
    espacement minimal entre les *départs* de requêtes sans bloquer les
    appels réseau eux-mêmes (ceux-ci peuvent donc se chevaucher).
    """

    def __init__(
        self,
        search_provider: SearchProvider,
        settings: Settings,
        cache: FileCache | None = None,
    ) -> None:
        self._search_provider = search_provider
        self._settings = settings
        self._cache = cache
        self._last_request_time: float = 0.0
        self._rate_limit_lock = threading.Lock()

    def find(self, restaurant: Restaurant) -> str | None:
        """Retourne l'URL Instagram la plus probable, ou None si non trouvée."""

        # Déjà fourni par OSM (contact:instagram) : pas besoin de chercher.
        if restaurant.instagram_url:
            return to_profile_url(restaurant.instagram_url) or restaurant.instagram_url

        cache_key = f"instagram:{restaurant.osm_id}:{restaurant.name}:{restaurant.city}"
        if self._cache is not None:
            cached = self._cache.get(cache_key)
            if cached is not None:
                return cached or None

        result = self._search_instagram(restaurant)

        if self._cache is not None:
            self._cache.set(cache_key, result or "")

        return result

    def _search_instagram(self, restaurant: Restaurant) -> str | None:
        best_handle: str | None = None
        best_score = -1

        for query in self._build_queries(restaurant):
            self._respect_rate_limit()
            try:
                results = self._search_provider.search(
                    query, max_results=self._settings.instagram_max_search_results
                )
            except SearchProviderError as exc:
                logger.warning(
                    "Recherche Instagram impossible pour %r (%r) : %s",
                    restaurant.name,
                    query,
                    exc,
                )
                continue

            handle, score = self._best_candidate(restaurant, results)
            if handle is not None and score > best_score:
                best_handle = handle
                best_score = score

            # Dès qu'un profil dépasse le seuil, on arrête (objectif = Instagram).
            if best_handle is not None and best_score >= self._settings.instagram_match_threshold:
                break

        if best_handle is None or best_score < self._settings.instagram_match_threshold:
            logger.debug(
                "Aucun profil Instagram fiable pour %r (meilleur score : %d).",
                restaurant.name,
                best_score,
            )
            return None

        logger.info(
            "Instagram trouvé pour %r : @%s (score %d).",
            restaurant.name,
            best_handle,
            best_score,
        )
        return to_profile_url(best_handle)

    @staticmethod
    def _build_queries(restaurant: Restaurant) -> list[str]:
        """Plusieurs formulations pour maximiser le taux de trouvaille Instagram."""

        name = restaurant.name.strip()
        city = restaurant.city.strip()
        return [
            # La plus efficace pour remonter des profils Instagram.
            f'site:instagram.com "{name}" {city}',
            f'"{name}" {city} Instagram',
            f"{name} {city} restaurant Instagram",
        ]

    def _best_candidate(
        self, restaurant: Restaurant, results: list[SearchResult]
    ) -> tuple[str | None, int]:
        best_handle: str | None = None
        best_score = -1

        for result in results:
            handle = extract_handle(result.url)
            if handle is None:
                continue

            candidate_text = (
                f"{result.title} {result.snippet} "
                f"{handle.replace('.', ' ').replace('_', ' ')}"
            )
            score = score_candidate(restaurant.name, candidate_text)

            # Bonus si le handle contient le nom de la ville (ex: lesafari_nice).
            city_token = restaurant.city.strip().lower().replace(" ", "")
            if city_token and city_token in handle.lower().replace("_", "").replace(".", ""):
                score = min(100, score + 8)

            if score > best_score:
                best_score = score
                best_handle = handle

        return best_handle, best_score

    def _respect_rate_limit(self) -> None:
        """Espace les départs de requêtes d'au moins `instagram_search_delay_seconds`."""

        with self._rate_limit_lock:
            elapsed = time.monotonic() - self._last_request_time
            wait_time = self._settings.instagram_search_delay_seconds - elapsed
            if wait_time > 0:
                time.sleep(wait_time)
            self._last_request_time = time.monotonic()


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/enrichment/instagram_followers.py
"""Client de lecture du nombre de followers d'un profil Instagram public.

Instagram ne propose pas d'API publique gratuite pour cela. On lit la page
profil web (comme un navigateur) et on extrait `follower_count` du JSON
embarqué. Fragile si Meta change le HTML, mais isolé derrière une interface
remplaçable.
"""

from __future__ import annotations

import logging
import re
import threading
import time

import requests

from restaurant_finder.cache import FileCache
from restaurant_finder.config import Settings
from restaurant_finder.enrichment.instagram_normalize import extract_handle

logger = logging.getLogger(__name__)

_FOLLOWER_COUNT_PATTERN = re.compile(r'"follower_count"\s*:\s*(\d+)')
_BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like Mac OS X) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 "
        "Mobile/15E148 Safari/604.1"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
}


class InstagramFollowerClient:
    """Récupère le nombre de followers d'un compte Instagram public."""

    def __init__(
        self,
        settings: Settings,
        cache: FileCache | None = None,
        session: requests.Session | None = None,
    ) -> None:
        self._settings = settings
        self._cache = cache
        self._session = session or requests.Session()
        self._session.headers.update(_BROWSER_HEADERS)
        self._lock = threading.Lock()
        self._last_request_time = 0.0
        self._session_warmed = False

    def get_follower_count(self, instagram_url_or_handle: str) -> int | None:
        """Retourne le nombre de followers, ou None si illisible / privé / bloqué."""

        handle = extract_handle(instagram_url_or_handle)
        if handle is None:
            return None

        cache_key = f"instagram_followers:{handle.lower()}"
        if self._cache is not None:
            cached = self._cache.get(cache_key)
            if cached is not None:
                return int(cached) if cached != "" else None

        count = self._fetch_follower_count(handle)

        if self._cache is not None:
            # On cache aussi les échecs (""), pour ne pas retaper Instagram en boucle.
            self._cache.set(cache_key, count if count is not None else "")

        return count

    def _fetch_follower_count(self, handle: str) -> int | None:
        with self._lock:
            self._respect_rate_limit()
            try:
                self._ensure_session()
                response = self._session.get(
                    f"https://www.instagram.com/{handle}/",
                    timeout=self._settings.request_timeout_seconds,
                )
                response.raise_for_status()
            except requests.RequestException as exc:
                logger.warning("Impossible de lire le profil @%s : %s", handle, exc)
                return None

            match = _FOLLOWER_COUNT_PATTERN.search(response.text)
            if not match:
                logger.warning(
                    "Nombre de followers introuvable pour @%s (page bloquée ou profil privé).",
                    handle,
                )
                return None

            count = int(match.group(1))
            logger.info("@%s : %d follower(s).", handle, count)
            return count

    def _ensure_session(self) -> None:
        if self._session_warmed:
            return
        try:
            self._session.get(
                "https://www.instagram.com/",
                timeout=self._settings.request_timeout_seconds,
            )
        except requests.RequestException as exc:
            logger.debug("Warm-up Instagram échoué : %s", exc)
        self._session_warmed = True

    def _respect_rate_limit(self) -> None:
        elapsed = time.monotonic() - self._last_request_time
        wait_time = self._settings.instagram_followers_delay_seconds - elapsed
        if wait_time > 0:
            time.sleep(wait_time)
        self._last_request_time = time.monotonic()


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/export/base.py
"""Interface abstraite d'un exporteur de résultats.

Ajouter un nouveau format d'export (JSON, Google Sheets, base de
données...) se fait en implémentant ce contrat, sans toucher au reste
de l'application (principe ouvert/fermé).
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from pathlib import Path
from typing import ClassVar

from restaurant_finder.domain.models import Restaurant


class Exporter(ABC):
    """Contrat pour tout exporteur de résultats."""

    file_extension: ClassVar[str]

    @abstractmethod
    def export(self, restaurants: list[Restaurant], destination: Path) -> Path:
        """Écrit `restaurants` dans un fichier et retourne le chemin final."""
        raise NotImplementedError

    def with_extension(self, destination: Path) -> Path:
        """Garantit que `destination` porte bien l'extension attendue."""

        if destination.suffix.lower() == f".{self.file_extension}":
            return destination
        return destination.with_suffix(f".{self.file_extension}")


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/export/csv_exporter.py
"""Export des résultats au format CSV."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

from restaurant_finder.domain.models import Restaurant
from restaurant_finder.exceptions import ExportError
from restaurant_finder.export.base import Exporter


class CsvExporter(Exporter):
    """Exporte une liste de restaurants vers un fichier CSV (UTF-8 avec BOM)."""

    file_extension = "csv"

    def export(self, restaurants: list[Restaurant], destination: Path) -> Path:
        destination = self.with_extension(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)

        rows = [restaurant.to_export_row() for restaurant in restaurants]
        dataframe = pd.DataFrame(rows, columns=["Nom", "Instagram", "Adresse", "Ville", "Catégorie"])

        try:
            # encoding="utf-8-sig" pour qu'Excel affiche correctement les accents.
            dataframe.to_csv(destination, index=False, encoding="utf-8-sig")
        except OSError as exc:
            raise ExportError(f"Impossible d'écrire le CSV {destination} : {exc}") from exc

        return destination


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/export/excel_exporter.py
"""Export des résultats au format Excel (.xlsx)."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

from restaurant_finder.domain.models import Restaurant
from restaurant_finder.exceptions import ExportError
from restaurant_finder.export.base import Exporter

_COLUMN_WIDTHS = {"Nom": 32, "Instagram": 24, "Adresse": 34, "Ville": 18, "Catégorie": 16}


class ExcelExporter(Exporter):
    """Exporte une liste de restaurants vers un classeur Excel (.xlsx)."""

    file_extension = "xlsx"

    def export(self, restaurants: list[Restaurant], destination: Path) -> Path:
        destination = self.with_extension(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)

        rows = [restaurant.to_export_row() for restaurant in restaurants]
        dataframe = pd.DataFrame(rows, columns=list(_COLUMN_WIDTHS.keys()))

        try:
            with pd.ExcelWriter(destination, engine="openpyxl") as writer:
                dataframe.to_excel(writer, index=False, sheet_name="Restaurants")
                self._autosize_columns(writer)
        except OSError as exc:
            raise ExportError(f"Impossible d'écrire le fichier Excel {destination} : {exc}") from exc

        return destination

    @staticmethod
    def _autosize_columns(writer: pd.ExcelWriter) -> None:
        worksheet = writer.sheets["Restaurants"]
        for index, width in enumerate(_COLUMN_WIDTHS.values(), start=1):
            column_letter = worksheet.cell(row=1, column=index).column_letter
            worksheet.column_dimensions[column_letter].width = width


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/services/restaurant_finder_service.py
"""Service d'orchestration : le seul point d'entrée de la logique métier.

`RestaurantFinderService` compose une `RestaurantSource` (récupération
des établissements), un `InstagramFinder` optionnel (enrichissement) et
un ensemble d'`Exporter`s. La CLI (ou toute autre couche de présentation
future, ex: une API) ne fait qu'appeler ce service : elle ne connaît ni
Overpass, ni DuckDuckGo, ni pandas.
"""

from __future__ import annotations

import logging
from collections.abc import Callable, Sequence
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from restaurant_finder.config import Settings
from restaurant_finder.domain.models import Restaurant
from restaurant_finder.enrichment.instagram_finder import InstagramFinder
from restaurant_finder.enrichment.instagram_followers import InstagramFollowerClient
from restaurant_finder.export.base import Exporter
from restaurant_finder.filtering.chain_filter import ChainRestaurantFilter
from restaurant_finder.geocoding.geo_math import PointQuery
from restaurant_finder.sources.base import RestaurantSource

logger = logging.getLogger(__name__)

#: Callback appelé après chaque restaurant enrichi : (fait, total).
ProgressCallback = Callable[[int, int], None]


class RestaurantFinderService:
    """Orchestre la recherche, l'enrichissement et l'export des restaurants."""

    def __init__(
        self,
        source: RestaurantSource,
        settings: Settings,
        instagram_finder: InstagramFinder | None = None,
        follower_client: InstagramFollowerClient | None = None,
        chain_filter: ChainRestaurantFilter | None = None,
    ) -> None:
        self._source = source
        self._settings = settings
        self._instagram_finder = instagram_finder
        self._follower_client = follower_client
        self._chain_filter = chain_filter or ChainRestaurantFilter()

    def find_restaurants(
        self,
        cities: Sequence[str] | None = None,
        near_points: Sequence[PointQuery] | None = None,
        categories: Sequence[str] | None = None,
        limit: int | None = None,
        enrich_instagram: bool = True,
        exclude_chains: bool = True,
        on_progress: ProgressCallback | None = None,
    ) -> list[Restaurant]:
        """Exécute le pipeline complet et retourne la liste des restaurants.

        Il faut fournir `cities`, `near_points`, ou les deux : les résultats
        de chaque ville et de chaque point sont fusionnés et dédupliqués par
        identifiant OSM. Ceci permet de combiner plusieurs villes et
        plusieurs lieux "pingués" (chacun avec son propre rayon) en une
        seule recherche étendue.
        """

        if not cities and not near_points:
            raise ValueError(
                "Il faut fournir au moins une ville (cities) ou un point (near_points)."
            )

        categories = tuple(categories) if categories else self._settings.default_categories

        restaurants: list[Restaurant] = []
        for city in cities or []:
            restaurants.extend(self._source.find_restaurants(city, categories))
        if near_points:
            restaurants.extend(
                self._source.find_restaurants_near_points(near_points, categories)
            )
        restaurants = self._dedupe_by_osm_id(restaurants)

        if exclude_chains:
            restaurants, excluded = self._chain_filter.exclude_chains(restaurants)
            if excluded:
                logger.info(
                    "%d enseigne(s)/franchise(s) exclue(s) — %d indépendant(s) restant(s).",
                    excluded,
                    len(restaurants),
                )

        if limit is not None:
            restaurants = restaurants[:limit]

        if enrich_instagram and self._instagram_finder is not None and restaurants:
            restaurants = self.enrich_with_instagram(restaurants, on_progress)

        return restaurants

    @staticmethod
    def _dedupe_by_osm_id(restaurants: list[Restaurant]) -> list[Restaurant]:
        """Fusionne les résultats de plusieurs recherches (ville + points GPS)."""

        seen: set[str] = set()
        deduped: list[Restaurant] = []
        for restaurant in restaurants:
            if restaurant.osm_id in seen:
                continue
            seen.add(restaurant.osm_id)
            deduped.append(restaurant)
        return deduped

    def enrich_with_instagram(
        self,
        restaurants: list[Restaurant],
        on_progress: ProgressCallback | None = None,
    ) -> list[Restaurant]:
        """Recherche Instagram, puis filtre selon le nombre de followers."""

        if self._instagram_finder is None:
            logger.warning("Enrichissement Instagram demandé mais aucun finder configuré.")
            return restaurants

        total = len(restaurants)
        completed = 0

        max_workers = max(1, self._settings.instagram_search_max_workers)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_restaurant = {
                executor.submit(self._resolve_instagram, restaurant): restaurant
                for restaurant in restaurants
            }

            for future in as_completed(future_to_restaurant):
                restaurant = future_to_restaurant[future]
                try:
                    url, followers = future.result()
                    restaurant.instagram_url = url
                    restaurant.instagram_followers = followers
                except Exception:  # noqa: BLE001 - on ne bloque jamais le pipeline
                    logger.exception(
                        "Erreur inattendue lors de la recherche Instagram pour %r.",
                        restaurant.name,
                    )
                    restaurant.instagram_url = None
                    restaurant.instagram_followers = None
                finally:
                    completed += 1
                    if on_progress is not None:
                        on_progress(completed, total)

        return restaurants

    def _resolve_instagram(self, restaurant: Restaurant) -> tuple[str | None, int | None]:
        assert self._instagram_finder is not None
        url = self._instagram_finder.find(restaurant)
        if url is None:
            return None, None

        if (
            not self._settings.instagram_filter_by_followers
            or self._follower_client is None
        ):
            return url, None

        followers = self._follower_client.get_follower_count(url)
        if followers is None:
            if self._settings.instagram_exclude_unknown_followers:
                logger.info(
                    "Instagram @%s exclu : nombre de followers illisible.",
                    extract_handle_safe(url),
                )
                return None, None
            return url, None

        if followers >= self._settings.instagram_max_followers:
            handle = extract_handle_safe(url)
            logger.info(
                "Instagram @%s exclu : %d followers (>= %d).",
                handle,
                followers,
                self._settings.instagram_max_followers,
            )
            return None, followers

        return url, followers

    @staticmethod
    def export(
        restaurants: list[Restaurant],
        exporters: Sequence[Exporter],
        destination: Path,
    ) -> list[Path]:
        """Exporte `restaurants` avec chaque exporteur fourni, vers `destination`."""

        return [exporter.export(restaurants, destination) for exporter in exporters]


def extract_handle_safe(url: str) -> str:
    from restaurant_finder.enrichment.instagram_normalize import extract_handle

    return extract_handle(url) or "?"


In [ ]:
%%writefile /content/restaurant_finder/src/restaurant_finder/bootstrap.py
"""Composition root : câble les implémentations concrètes entre elles.

Isoler ce câblage dans un seul module permet de garder `cli.py` focalisé
sur la présentation, et de changer une implémentation (ex: un autre
`SearchProvider`) sans toucher à la logique métier ni à la CLI.
"""

from __future__ import annotations

from restaurant_finder.cache import FileCache
from restaurant_finder.config import Settings
from restaurant_finder.enrichment.instagram_finder import InstagramFinder
from restaurant_finder.enrichment.instagram_followers import InstagramFollowerClient
from restaurant_finder.enrichment.search_providers.ddgs_provider import DdgsSearchProvider
from restaurant_finder.geocoding.location_parser import LocationInputParser
from restaurant_finder.geocoding.nominatim_client import NominatimGeocoder
from restaurant_finder.http.client import build_http_session
from restaurant_finder.services.restaurant_finder_service import RestaurantFinderService
from restaurant_finder.sources.overpass_source import OverpassRestaurantSource


def build_location_parser(settings: Settings) -> LocationInputParser:
    """Construit un `LocationInputParser` (coordonnées / liens Google Maps)."""

    return LocationInputParser(session=build_http_session(settings), settings=settings)


def build_service(settings: Settings, enable_instagram: bool = True) -> RestaurantFinderService:
    """Construit un `RestaurantFinderService` entièrement câblé et prêt à l'emploi."""

    session = build_http_session(settings)
    # Overpass : zéro retry HTTP — un échec doit basculer immédiatement sur le miroir suivant.
    overpass_session = build_http_session(settings, max_retries=0)
    cache = FileCache(
        cache_dir=settings.cache_dir,
        ttl_seconds=settings.cache_ttl_seconds,
        enabled=settings.cache_enabled,
    )

    geocoder = NominatimGeocoder(session=session, settings=settings, cache=cache)
    source = OverpassRestaurantSource(
        session=overpass_session, settings=settings, geocoder=geocoder
    )

    instagram_finder: InstagramFinder | None = None
    follower_client: InstagramFollowerClient | None = None
    if enable_instagram and settings.instagram_search_enabled:
        # ddgs (sans clé API) : plus fiable que le scraping HTML DuckDuckGo.
        search_provider = DdgsSearchProvider(settings=settings)
        instagram_finder = InstagramFinder(
            search_provider=search_provider, settings=settings, cache=cache
        )
        if settings.instagram_filter_by_followers:
            follower_client = InstagramFollowerClient(settings=settings, cache=cache)

    return RestaurantFinderService(
        source=source,
        settings=settings,
        instagram_finder=instagram_finder,
        follower_client=follower_client,
    )


## Étape 4 — Configuration et import du package

On ajoute `src/` au chemin Python (`sys.path`) pour pouvoir importer `restaurant_finder` comme un package normal, puis on définit la configuration (équivalent du fichier `.env` du projet original, mais via des variables d'environnement directement dans Colab).


In [ ]:
import os
import sys

# Rend le package `restaurant_finder` importable, comme s'il était installé.
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# --- Configuration (équivalent du fichier .env du projet, via variables d'environnement) ---
# IMPORTANT : Nominatim exige un User-Agent identifiable (indique un contact réel).
# 👉 Remplace l'adresse e-mail ci-dessous par la tienne avant de lancer une recherche.
os.environ["RF_USER_AGENT"] = "RestaurantFinderColab/0.1 (+contact: votre-email@example.com)"

# Le disque de la VM Colab est éphémère (tout est perdu à la fin du runtime) :
# on désactive le cache disque pour éviter d'accumuler des fichiers inutiles.
os.environ["RF_CACHE_ENABLED"] = "false"

# Pour aller plus vite/plus loin, tu peux surcharger d'autres réglages ici, ex :
# os.environ["RF_INSTAGRAM_MATCH_THRESHOLD"] = "55"
# os.environ["RF_OVERPASS_TIMEOUT_SECONDS"] = "60"

from restaurant_finder.bootstrap import build_location_parser, build_service
from restaurant_finder.config import DEFAULT_CATEGORY_LABELS, get_settings
from restaurant_finder.export import EXPORTERS
from restaurant_finder.geocoding.geo_math import PointQuery
from restaurant_finder.utils.logging import setup_logging

setup_logging(verbose=False)

print("✅ Package restaurant_finder importé et configuré. Catégories disponibles :")
print(", ".join(DEFAULT_CATEGORY_LABELS))


## Étape 5 — La fonction `rechercher_restaurants(...)`

Cette fonction reproduit **exactement** le pipeline de la commande `restaurant-finder search <ville>` de la CLI :

1. Recherche des établissements (OpenStreetMap / Overpass) pour la ville (ou les lieux GPS via `near=`).
2. Exclusion des grandes enseignes/franchises (sauf si `include_chains=True`).
3. Recherche du profil Instagram de chaque établissement (sauf si `no_instagram=True`).
4. Filtrage par nombre de followers Instagram (`max_followers`, défaut 1000).
5. Aperçu du tableau de résultats directement dans le notebook.
6. Export CSV/Excel + téléchargement automatique sur ton téléphone/ordinateur.


In [ ]:
from pathlib import Path

import pandas as pd

from restaurant_finder.exceptions import LocationParsingError, RestaurantFinderError


def rechercher_restaurants(
    ville: str | None = None,
    near: list[str] | None = None,
    radius: float = 800.0,
    categories: list[str] | None = None,
    limit: int | None = None,
    formats: list[str] | None = None,
    no_instagram: bool = False,
    only_with_instagram: bool = False,
    include_chains: bool = False,
    max_followers: int = 1000,
    keep_unknown_followers: bool = False,
    telecharger: bool = True,
) -> pd.DataFrame:
    """Recherche des restaurants et exporte le résultat en CSV/Excel.

    Équivalent Colab de : `restaurant-finder search "<ville>" [options]`.

    Args:
        ville: nom de la ville à rechercher, ex: "Nice". Optionnel si `near` est fourni.
        near: liste de lieux "pingués" sur Google Maps — coordonnées collées
            (ex: "43.6970, 7.2707") ou lien Google Maps complet. Étend la
            recherche à un ou plusieurs lieux précis en plus (ou à la place)
            de `ville`.
        radius: rayon en mètres autour de chaque `near` (défaut 800m).
        categories: catégories OSM à inclure, parmi restaurant/cafe/fast_food/
            bar/pub/biergarten. Par défaut : toutes.
        limit: nombre maximal de restaurants à retourner.
        formats: formats d'export, parmi "csv" et "xlsx" (défaut : les deux).
        no_instagram: désactive la recherche Instagram (plus rapide).
        only_with_instagram: n'exporte que les établissements avec un Instagram trouvé.
        include_chains: inclut aussi les grandes enseignes/franchises (exclues par défaut).
        max_followers: exclut les comptes Instagram avec au moins ce nombre de followers.
        keep_unknown_followers: garde un Instagram même si le nombre de followers
            n'a pas pu être lu (exclus par défaut).
        telecharger: si True (défaut), déclenche le téléchargement automatique
            des fichiers exportés dans le navigateur (fonctionne aussi sur mobile).

    Returns:
        Le DataFrame pandas des résultats (déjà affiché dans le notebook).
    """

    formats = formats or ["csv", "xlsx"]

    if no_instagram and only_with_instagram:
        raise ValueError("no_instagram et only_with_instagram sont incompatibles.")
    if not ville and not near:
        raise ValueError("Précise une ville (ex: 'Nice') ou au moins un lieu via near=[...].")

    settings = get_settings()
    settings.instagram_max_followers = max_followers
    settings.instagram_exclude_unknown_followers = not keep_unknown_followers

    near_points: list[PointQuery] = []
    if near:
        location_parser = build_location_parser(settings)
        for raw_location in near:
            try:
                latitude, longitude = location_parser.parse(raw_location)
            except LocationParsingError as exc:
                raise ValueError(f"Lieu invalide ({raw_location!r}) : {exc}") from exc
            near_points.append(PointQuery(latitude, longitude, radius))

    service = build_service(settings, enable_instagram=not no_instagram)

    if ville:
        print(f"🔎 Recherche des établissements à {ville}...")
    if near_points:
        points_label = ", ".join(f"({p.latitude:.5f}, {p.longitude:.5f})" for p in near_points)
        print(f"🔎 Recherche autour de {points_label} (rayon {radius:.0f}m)...")
    if not include_chains:
        print("ℹ️ Filtre actif : enseignes / franchises exclues.")
    if not no_instagram:
        print(f"ℹ️ Filtre Instagram : moins de {max_followers} followers.")

    try:
        restaurants = service.find_restaurants(
            cities=[ville] if ville else None,
            near_points=near_points or None,
            categories=categories or None,
            limit=limit,
            enrich_instagram=not no_instagram,
            exclude_chains=not include_chains,
        )
    except RestaurantFinderError as exc:
        raise RuntimeError(f"Erreur : {exc}") from exc

    if not restaurants:
        print("⚠️ Aucun établissement trouvé pour cette recherche.")
        return pd.DataFrame(columns=["Nom", "Instagram", "Adresse", "Ville", "Catégorie"])

    print(
        f"✅ {len(restaurants)} établissement(s) indépendant(s) retenu(s)."
        if not include_chains
        else f"✅ {len(restaurants)} établissement(s) trouvé(s)."
    )

    if not no_instagram:
        with_instagram = sum(1 for item in restaurants if item.instagram_url)
        print(f"📸 {with_instagram}/{len(restaurants)} profil(s) Instagram trouvé(s).")

        if only_with_instagram:
            restaurants = [item for item in restaurants if item.instagram_url]
            if not restaurants:
                print("⚠️ Aucun profil Instagram trouvé : rien à exporter.")
                return pd.DataFrame(columns=["Nom", "Instagram", "Adresse", "Ville", "Catégorie"])
            print(f"✅ Export filtré : {len(restaurants)} établissement(s) avec Instagram.")

    # Aperçu direct dans le notebook (équivalent du tableau Rich de la CLI).
    dataframe = pd.DataFrame([r.to_export_row() for r in restaurants])
    try:
        from IPython.display import display

        display(dataframe)
    except ImportError:
        print(dataframe.to_string(index=False))

    # Export CSV/Excel, comme `--output output/<ville>` en CLI.
    output_name = (ville or "recherche").strip().lower().replace(" ", "_") or "recherche"
    output_path = Path("/content/output") / output_name
    exporters = [EXPORTERS[fmt] for fmt in formats]
    exported_paths = [exporter.export(restaurants, output_path) for exporter in exporters]

    print("\n📁 Export terminé :")
    for path in exported_paths:
        print(f"  • {path}")

    if telecharger:
        try:
            from google.colab import files  # disponible uniquement sur Colab

            for path in exported_paths:
                files.download(str(path))
        except ImportError:
            print("(Téléchargement automatique disponible uniquement dans l'environnement Colab.)")

    return dataframe


print("✅ Fonction rechercher_restaurants(...) prête à l'emploi.")


## Étape 6 — Exemple d'utilisation

Exécute la cellule ci-dessous (ou ajoute-en une nouvelle) pour lancer une recherche. Les fichiers `nice.csv` et `nice.xlsx` seront proposés au téléchargement automatiquement.

Autres exemples possibles :

```python
# Recherche rapide sans Instagram
rechercher_restaurants("Lyon", no_instagram=True)

# Uniquement les établissements avec un Instagram trouvé, limité à 50
rechercher_restaurants("Bordeaux", limit=50, only_with_instagram=True)

# Recherche autour d'un lieu pingué sur Google Maps (coordonnées collées)
rechercher_restaurants(near=["43.6970, 7.2707"], radius=500)
```

In [ ]:
# Remplace "Nice" par la ville de ton choix.
resultats = rechercher_restaurants("Nice")
